# **Forgery Detection — Defactify Only (Binary: Real vs AI-Generated)**

Simplified from the original multi-source, multi-task pipeline. This version:
- Uses **only the Defactify dataset** (Hugging Face)
- Trains a **binary classifier** (`real` vs `ai_generated`) — no forgery-type head, no segmentation head
- Removed: CASIA v2, SROIE, synthetic tampering generation, masks, multi-class type head, segmentation head

In [ ]:
!pip install timm scikit-learn datasets huggingface_hub --quiet

**Load Defactify dataset**

In [ ]:
from datasets import load_dataset
from huggingface_hub import login
# from google.colab import userdata
# login(token=userdata.get('HF_TOKEN'))  # uncomment if dataset requires auth

ai_df = load_dataset(
    "Rajarshi-Roy-research/Defactify_Image_Dataset",
    split="train"
)  # ~42k images available

print(ai_df)
print(ai_df[0])

In [ ]:
# Sanity check labels
labels_a = [ai_df[i]['Label_A'] for i in range(100)]
print("Label_A unique:", set(labels_a))

for i in range(5):
    sample = ai_df[i]
    print(f"Sample {i}: Label_A={sample['Label_A']} Caption={sample['Caption'][:100]}")

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())

## Build master label dataframe (Defactify only)

In [ ]:
import pandas as pd
from tqdm import tqdm

records = []

# Balance real vs ai_generated — adjust target based on how much data you want
target = 2500  # per class; raise this for a larger training set (up to ~21k each if class-balanced in source)

real_count = 0
ai_count   = 0

for i, sample in enumerate(tqdm(ai_df, desc="processing")):
    label = sample['Label_A']  # 0 = real, 1 = AI-generated

    if label == 0 and real_count < target:
        records.append({
            "file_id"    : f"defactify_real_{i:05d}",
            "image_path" : f"defactify_index_{i}",
            "is_forged"  : 0,
            "source_dataset": "DEFACTIFY"
        })
        real_count += 1

    elif label == 1 and ai_count < target:
        records.append({
            "file_id"    : f"defactify_ai_{i:05d}",
            "image_path" : f"defactify_index_{i}",
            "is_forged"  : 1,
            "source_dataset": "DEFACTIFY"
        })
        ai_count += 1

    if real_count >= target and ai_count >= target:
        break

df = pd.DataFrame(records)
df.to_csv("/content/master_labels.csv", index=False)

print(f"Real added : {real_count}")
print(f"AI added   : {ai_count}")
print(f"\nTotal records : {len(df)}")
print(df.is_forged.value_counts())

## Dataset sanity check — view a few samples

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def load_image(image_path):
    image_path = str(image_path)
    if image_path.startswith("defactify_index_"):
        i = int(image_path.split("_")[-1])
        image = ai_df[i]["Image"].convert("RGB")
        return np.array(image)
    raise ValueError(f"Unexpected path format: {image_path}")

real_sample = df[df.is_forged == 0].iloc[0]
ai_sample   = df[df.is_forged == 1].iloc[0]

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(load_image(real_sample["image_path"]))
axes[0].set_title("Real")
axes[0].axis("off")

axes[1].imshow(load_image(ai_sample["image_path"]))
axes[1].set_title("AI Generated")
axes[1].axis("off")

plt.tight_layout()
plt.savefig("/content/dataset_check.png")
plt.show()
print("Sanity check complete \u2713")

## Imports + device setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms as transforms
import timm
import numpy as np
import pandas as pd
from PIL import Image
import io
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

## Dataset Loader

No mask, no forgery-type label — just image + binary label.

In [ ]:
class ForgeryDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        try:
            # All paths are defactify_index_N in this simplified version
            i = int(str(row["image_path"]).split("_")[-1])
            image = ai_df[i]["Image"].convert("RGB")
            image = np.array(image)
            image = Image.fromarray(image)

            if self.transform:
                image = self.transform(image)

        except Exception:
            image = torch.zeros(3, 224, 224)

        label = torch.tensor(row["is_forged"], dtype=torch.float32)

        return image, label

## Model Architecture

**Binary classifier only.** `type_head` and `seg_head` removed — there is no forgery-type label or mask in the Defactify-only dataset, so those heads had nothing to learn from anyway.

In [ ]:
BACKBONE_NAME = "efficientnet_b4"
# Other options: "resnet50d", "convnext_small", "swin_small_patch4_window7_224"

class ForgeryDetector(nn.Module):
    def __init__(self, backbone_name=BACKBONE_NAME):
        super().__init__()

        self.backbone = timm.create_model(
            backbone_name,
            pretrained=True,
            features_only=True,
        )

        with torch.no_grad():
            dummy    = torch.zeros(1, 3, 224, 224)
            features = self.backbone(dummy)
            channels = features[-1].shape[1]
            print(f"Backbone        : {backbone_name}")
            print(f"Output channels : {channels}")

        # ELA branch — still useful for AI-generated detection (compression/artifact cues)
        self.ela_conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(),
        )

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier  = nn.Sequential(
            nn.Flatten(),
            nn.Linear(channels + 64, 512), nn.BatchNorm1d(512),
            nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(256, 1)  # logit -> sigmoid -> real / ai_generated
        )

    def compute_ela(self, x, quality=90):
        ela_imgs = []
        for img in x:
            try:
                img_np = (img.permute(1, 2, 0).cpu().float().numpy() * 255).astype(np.uint8)
                pil_img = Image.fromarray(img_np)
                buf = io.BytesIO()
                pil_img.save(buf, format="JPEG", quality=quality)
                buf.seek(0)
                recompressed = np.array(Image.open(buf)).astype(np.float32)
                ela = np.abs(img_np.astype(np.float32) - recompressed)
                if ela.max() > 0:
                    ela = ela / ela.max()
                ela_imgs.append(torch.tensor(ela).permute(2, 0, 1).float())
            except Exception:
                ela_imgs.append(torch.zeros(3, 224, 224))
        return torch.stack(ela_imgs).to(x.device)

    def forward(self, x):
        features  = self.backbone(x)
        last_feat = features[-1]

        ela      = self.compute_ela(x)
        ela_feat = self.ela_conv(ela)

        pooled   = self.global_pool(last_feat)
        pooled   = pooled.view(pooled.size(0), -1)
        combined = torch.cat([pooled, ela_feat], dim=1)

        cls_out = self.classifier(combined).squeeze(1)
        return cls_out


import gc
gc.collect()
torch.cuda.empty_cache()

model = ForgeryDetector(BACKBONE_NAME).to(DEVICE)
print(f"\nModel ready \u2713")
print(f"Parameters : {sum(p.numel() for p in model.parameters()):,}")

## Transforms + Train/Val Split + DataLoaders

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.2, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

df = pd.read_csv("/content/master_labels.csv")
print(df["is_forged"].value_counts())

train_df, val_df = train_test_split(
    df, test_size=0.15,
    stratify=df["is_forged"],
    random_state=42
)

print(f"\nTrain : {len(train_df)}")
print(f"Val   : {len(val_df)}")
print(f"\nVal distribution:\n{val_df['is_forged'].value_counts()}")

train_dataset = ForgeryDataset(train_df, transform=transform_train)
val_dataset   = ForgeryDataset(val_df,   transform=transform_val)

train_loader = DataLoader(
    train_dataset, batch_size=16, shuffle=True,
    num_workers=2, pin_memory=True, drop_last=True
)
val_loader = DataLoader(
    val_dataset, batch_size=16, shuffle=False,
    num_workers=2, pin_memory=True
)

print(f"\nTrain batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")

## Training Loop

Single loss term now — no `type_loss`, no `seg_loss`, no masking logic needed.

In [ ]:
from torch.cuda.amp import autocast, GradScaler

scaler             = GradScaler()
ACCUMULATION_STEPS = 4

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.85, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        bce   = nn.functional.binary_cross_entropy_with_logits(inputs, targets, reduction="none")
        pt    = torch.exp(-bce)
        focal = self.alpha * (1 - pt) ** self.gamma * bce
        return focal.mean()

cls_criterion = FocalLoss(alpha=0.85, gamma=2.0)


def train_epoch(model, loader, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    optimizer.zero_grad()

    for i, (images, labels) in enumerate(tqdm(loader, desc="Training")):
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        with autocast():
            cls_out = model(images)
            loss = cls_criterion(cls_out, labels) / ACCUMULATION_STEPS

        scaler.scale(loss).backward()

        if (i + 1) % ACCUMULATION_STEPS == 0 or (i + 1) == len(loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        total_loss += loss.item() * ACCUMULATION_STEPS
        preds       = (torch.sigmoid(cls_out) > 0.5).float()
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

        if i % 50 == 0:
            torch.cuda.empty_cache()

    return total_loss / len(loader), correct / total


def val_epoch(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validation"):
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            with autocast():
                cls_out = model(images)
                loss = cls_criterion(cls_out, labels)

            total_loss += loss.item()
            preds       = (torch.sigmoid(cls_out) > 0.5).float()
            correct    += (preds == labels).sum().item()
            total      += labels.size(0)

    return total_loss / len(loader), correct / total

print("Training functions ready \u2713")

## Optimizer + Scheduler

In [ ]:
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR

optimizer = optim.AdamW([
    {"params": model.backbone.parameters(),   "lr": 5e-5},
    {"params": model.classifier.parameters(), "lr": 2e-4},
    {"params": model.ela_conv.parameters(),   "lr": 2e-4},
], weight_decay=1e-4)

warmup_scheduler = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=3)
cosine_scheduler = CosineAnnealingLR(optimizer, T_max=15, eta_min=1e-6)

scheduler = SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[3]
)

WARMUP_EPOCHS = 3
print("Optimizer + scheduler ready \u2713")

## Run Training

In [ ]:
import gc
import os
from sklearn.metrics import confusion_matrix, classification_report
from google.colab import drive

drive.mount('/content/drive')
os.makedirs("/content/drive/MyDrive/forgery_detection_defactify", exist_ok=True)

EPOCHS     = 18
best_val   = 0
patience   = 5
no_improve = 0
history    = []

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_loss, train_acc = train_epoch(model, train_loader, optimizer)
    val_loss, val_acc      = val_epoch(model, val_loader)
    scheduler.step()

    print(f"Train loss: {train_loss:.4f}  acc: {train_acc:.4f}")
    print(f"Val   loss: {val_loss:.4f}  acc: {val_acc:.4f}")

    history.append({
        "epoch": epoch + 1, "train_loss": train_loss, "train_acc": train_acc,
        "val_loss": val_loss, "val_acc": val_acc
    })

    if val_acc > best_val:
        best_val   = val_acc
        no_improve = 0
        torch.save(
            model.state_dict(),
            "/content/drive/MyDrive/forgery_detection_defactify/model_weights.pth"
        )
        print(f"  \u2713 New best model saved (val_acc={val_acc:.4f})")
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

print(f"\nBest val accuracy: {best_val:.4f}")

## Evaluation — Confusion Matrix + Classification Report

In [ ]:
import seaborn as sns

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Evaluating"):
        images = images.to(DEVICE)
        with autocast():
            cls_out = model(images)
        preds = (torch.sigmoid(cls_out) > 0.5).float().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

cm     = confusion_matrix(all_labels, all_preds)
report = classification_report(
    all_labels, all_preds,
    target_names=["Real", "AI_Generated"],
    output_dict=True
)

print(classification_report(all_labels, all_preds, target_names=["Real", "AI_Generated"]))

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Real", "AI_Generated"],
            yticklabels=["Real", "AI_Generated"], ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix (Acc={report['accuracy']:.3f})")
plt.tight_layout()
plt.savefig("/content/confusion_matrix.png")
plt.show()

## Load Model For Prediction

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ForgeryDetector(BACKBONE_NAME).to(DEVICE)
model.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/forgery_detection_defactify/model_weights.pth",
        map_location=DEVICE
    )
)
model.eval()
print("Model loaded \u2713")

## Predict Function

No forgery-type output, no segmentation heatmap — binary verdict + confidence only.

In [ ]:
def load_image_for_predict(image_path):
    image_path = str(image_path)
    if image_path.startswith("defactify_index_"):
        i = int(image_path.split("_")[-1])
        image = ai_df[i]["Image"].convert("RGB")
        return np.array(image)
    else:
        import cv2
        image = cv2.imread(image_path)
        if image is None:
            raise FileNotFoundError(f"Image not found: {image_path}")
        return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


def predict(image_path, threshold=0.50):
    import cv2

    image_rgb = load_image_for_predict(image_path)
    image_rgb = cv2.resize(image_rgb, (224, 224))

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    tensor = transform(Image.fromarray(image_rgb)).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        with autocast():
            cls_out = model(tensor)

    confidence = float(torch.sigmoid(cls_out).item())
    confidence = max(0.0, min(1.0, confidence))
    is_ai      = confidence > threshold

    result_text  = "AI GENERATED" if is_ai else "REAL"
    result_color = "red" if is_ai else "green"

    plt.figure(figsize=(5, 5))
    plt.imshow(image_rgb)
    plt.axis("off")
    plt.title(f"{result_text}  |  Confidence: {confidence:.2%}", color=result_color, fontweight="bold")
    plt.tight_layout()
    plt.savefig("/content/prediction_result.png")
    plt.show()

    print(f"\n{'='*40}")
    print(f"  Result     : {result_text}")
    print(f"  Confidence : {confidence:.2%}")
    print(f"{'='*40}")

    return {"is_ai_generated": is_ai, "confidence": confidence}

## Test With Your Own Uploaded Image

In [ ]:
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f"testing uploaded image: {filename}")

result = predict(f"/content/{filename}")

## Test On Held-Out Dataset Samples

In [ ]:
real_sample = df[df["is_forged"] == 0].iloc[2]
res1 = predict(real_sample["image_path"])
print("\n" + "="*30)

ai_sample = df[df["is_forged"] == 1].iloc[5]
res2 = predict(ai_sample["image_path"])